# Auditoria da Gold — ENEM

Auditoria da camada Gold territorial do ENEM para o Pará, 1998–2025.

A auditoria verifica:
- integridade do manifesto e dos arquivos;
- schema canônico;
- cardinalidade Silver → Gold;
- referência territorial;
- integridade territorial dos registros;
- compatibilidade municipal entre `PARTICIPANTES` e `RESULTADOS` em 2024–2025;
- perfil de preenchimento das variáveis analíticas.


## 1 — Configuração e manifesto

In [1]:

# ============================================================
# 1. CONFIGURAÇÃO E MANIFESTO GOLD
# ============================================================

from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import display

# ------------------------------------------------------------
# CONFIGURAÇÃO — altere somente a UF
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/home/akel/PycharmProjects/Projeto_ENEM"
)

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

UF_PROCESSAMENTO = "PA"  # Ex.: PA, RR, AM, AC

ANO_INICIAL = 1998
ANO_FINAL = 2025

# Códigos oficiais das UFs segundo o IBGE
CODIGOS_UF = {
    "RO": "11", "AC": "12", "AM": "13", "RR": "14",
    "PA": "15", "AP": "16", "TO": "17", "MA": "21",
    "PI": "22", "CE": "23", "RN": "24", "PB": "25",
    "PE": "26", "AL": "27", "SE": "28", "BA": "29",
    "MG": "31", "ES": "32", "RJ": "33", "SP": "35",
    "PR": "41", "SC": "42", "RS": "43", "MS": "50",
    "MT": "51", "GO": "52", "DF": "53"
}

UF_PROCESSAMENTO = UF_PROCESSAMENTO.strip().upper()

if UF_PROCESSAMENTO not in CODIGOS_UF:
    raise ValueError(
        f"UF inválida: {UF_PROCESSAMENTO}"
    )

CODIGO_UF_PROCESSAMENTO = CODIGOS_UF[UF_PROCESSAMENTO]

ARQUIVO_MANIFESTO_GOLD = (
    METADATA_DIR
    / f"manifesto_gold_{UF_PROCESSAMENTO}.csv"
)

if not ARQUIVO_MANIFESTO_GOLD.exists():
    raise FileNotFoundError(
        f"Manifesto não encontrado: {ARQUIVO_MANIFESTO_GOLD}"
    )

# Detecta automaticamente separador CSV
manifesto_gold = pd.read_csv(
    ARQUIVO_MANIFESTO_GOLD,
    sep=None,
    engine="python",
    encoding="utf-8"
)

# Verificação das colunas mínimas necessárias
COLUNAS_MANIFESTO = {
    "uf", "ano", "base", "arquivo",
    "arquivo_gold", "join_individual"
}

faltantes = COLUNAS_MANIFESTO - set(manifesto_gold.columns)

if faltantes:
    raise ValueError(
        f"Colunas ausentes no manifesto: {sorted(faltantes)}"
    )

con = duckdb.connect()

print(f"UF selecionada: {UF_PROCESSAMENTO}")
print(f"Código IBGE da UF: {CODIGO_UF_PROCESSAMENTO}")
print(f"Período: {ANO_INICIAL}–{ANO_FINAL}")
print(f"Manifesto: {ARQUIVO_MANIFESTO_GOLD}")
print(f"Fontes registradas: {len(manifesto_gold)}")
print(f"Edições: {manifesto_gold['ano'].nunique()}")


UF selecionada: PA
Código IBGE da UF: 15
Período: 1998–2025
Manifesto: /home/akel/PycharmProjects/Projeto_ENEM/data/metadata/manifesto_gold_PA.csv
Fontes registradas: 30
Edições: 28


## 2 — Integridade do manifesto e dos arquivos

In [2]:

# ============================================================
# 2. INTEGRIDADE DO MANIFESTO E DOS ARQUIVOS
# ============================================================

assert manifesto_gold["uf"].astype(str).str.upper().eq(
    UF_PROCESSAMENTO
).all(), "O manifesto contém uma UF diferente da selecionada."

manifesto_gold["ano"] = pd.to_numeric(
    manifesto_gold["ano"], errors="raise"
).astype(int)

assert set(manifesto_gold["ano"]) == set(
    range(ANO_INICIAL, ANO_FINAL + 1)
), "O manifesto não cobre todos os anos esperados."

assert manifesto_gold["arquivo"].map(
    lambda p: Path(str(p)).exists()
).all(), "Existem arquivos Silver ausentes."

assert manifesto_gold["arquivo_gold"].map(
    lambda p: Path(str(p)).exists()
).all(), "Existem arquivos Gold ausentes."

fontes_por_ano = manifesto_gold.groupby("ano")["base"].count()

assert fontes_por_ano.loc[ANO_INICIAL:2023].eq(1).all(), (
    "Esperava-se uma fonte por ano entre 1998 e 2023."
)

for ano in (2024, 2025):
    bases = set(
        manifesto_gold.loc[
            manifesto_gold["ano"] == ano, "base"
        ].astype(str).str.lower()
    )

    assert bases == {"participantes", "resultados"}, (
        f"Bases inesperadas em {ano}: {bases}"
    )

assert manifesto_gold["join_individual"].astype(str).str.lower().isin(
    ["false", "0"]
).all(), "Há fontes marcadas com integração individual."

print("Manifesto e arquivos: validação concluída.")
print(f"Fontes verificadas: {len(manifesto_gold)}")
print(f"Anos cobertos: {manifesto_gold['ano'].nunique()}")


Manifesto e arquivos: validação concluída.
Fontes verificadas: 30
Anos cobertos: 28


## 3 — Schema físico

In [3]:

# ============================================================
# 3. CONTRATO CANÔNICO E SCHEMA FÍSICO DA GOLD
# ============================================================

COLUNAS_GOLD = [
    "NU_ANO",
    "TIPO_REFERENCIA_TERRITORIAL",
    "SG_UF_TERRITORIO",
    "CO_UF_TERRITORIO",
    "CO_MUNICIPIO_TERRITORIO",
    "NO_MUNICIPIO_TERRITORIO",
    "TP_SEXO",
    "TP_FAIXA_ETARIA",
    "ANO_CONCLUIU",
    "TP_PRESENCA",
    "TP_PRESENCA_CN",
    "TP_PRESENCA_CH",
    "TP_PRESENCA_LC",
    "TP_PRESENCA_MT",
    "NU_NOTA_CN",
    "NU_NOTA_CH",
    "NU_NOTA_LC",
    "NU_NOTA_MT",
    "NU_NOTA_OBJETIVA",
    "NU_NOTA_REDACAO",
    "TP_STATUS_REDACAO",
    "VL_PERC_COMP1",
    "VL_PERC_COMP2",
    "VL_PERC_COMP3",
    "VL_PERC_COMP4",
    "VL_PERC_COMP5",
    "CO_ESCOLA",
    "TP_ESCOLA",
    "TP_DEPENDENCIA_ADM_ESC",
]

TIPOS_CANONICOS_GOLD = {
    "NU_ANO": "INTEGER",
    "TIPO_REFERENCIA_TERRITORIAL": "VARCHAR",
    "SG_UF_TERRITORIO": "VARCHAR",
    "CO_UF_TERRITORIO": "VARCHAR",
    "CO_MUNICIPIO_TERRITORIO": "VARCHAR",
    "NO_MUNICIPIO_TERRITORIO": "VARCHAR",
    "TP_SEXO": "VARCHAR",
    "TP_FAIXA_ETARIA": "INTEGER",
    "ANO_CONCLUIU": "INTEGER",
    "TP_PRESENCA": "INTEGER",
    "TP_PRESENCA_CN": "INTEGER",
    "TP_PRESENCA_CH": "INTEGER",
    "TP_PRESENCA_LC": "INTEGER",
    "TP_PRESENCA_MT": "INTEGER",
    "NU_NOTA_CN": "DOUBLE",
    "NU_NOTA_CH": "DOUBLE",
    "NU_NOTA_LC": "DOUBLE",
    "NU_NOTA_MT": "DOUBLE",
    "NU_NOTA_OBJETIVA": "DOUBLE",
    "NU_NOTA_REDACAO": "DOUBLE",
    "TP_STATUS_REDACAO": "VARCHAR",
    "VL_PERC_COMP1": "DOUBLE",
    "VL_PERC_COMP2": "DOUBLE",
    "VL_PERC_COMP3": "DOUBLE",
    "VL_PERC_COMP4": "DOUBLE",
    "VL_PERC_COMP5": "DOUBLE",
    "CO_ESCOLA": "VARCHAR",
    "TP_ESCOLA": "INTEGER",
    "TP_DEPENDENCIA_ADM_ESC": "INTEGER",
}

assert len(COLUNAS_GOLD) == 29
assert len(TIPOS_CANONICOS_GOLD) == 29
assert set(COLUNAS_GOLD) == set(TIPOS_CANONICOS_GOLD)

auditoria_schema = []

for _, fonte in manifesto_gold.iterrows():
    arquivo_gold = str(fonte["arquivo_gold"])

    schema = con.execute(
        """
        DESCRIBE
        SELECT *
        FROM read_parquet(?, hive_partitioning = false)
        """,
        [arquivo_gold]
    ).df()

    colunas_reais = schema["column_name"].tolist()
    tipos_reais = schema["column_type"].str.upper().tolist()

    tipos_esperados = [
        TIPOS_CANONICOS_GOLD[coluna]
        for coluna in COLUNAS_GOLD
    ]

    colunas_ok = colunas_reais == COLUNAS_GOLD
    tipos_ok = tipos_reais == tipos_esperados

    auditoria_schema.append({
        "ano": int(fonte["ano"]),
        "base": fonte["base"],
        "n_colunas": len(colunas_reais),
        "colunas_ok": colunas_ok,
        "tipos_ok": tipos_ok,
        "schema_ok": colunas_ok and tipos_ok,
        "colunas_reais": colunas_reais,
    })

auditoria_schema = pd.DataFrame(auditoria_schema)

display(
    auditoria_schema[
        [
            "ano", "base", "n_colunas",
            "colunas_ok", "tipos_ok", "schema_ok"
        ]
    ]
)

print(
    f"Arquivos com esquema correto: "
    f"{auditoria_schema['schema_ok'].sum()}/"
    f"{len(auditoria_schema)}"
)

# Mostra a estrutura real do primeiro arquivo com divergência
divergencias = auditoria_schema[
    ~auditoria_schema["schema_ok"]
]

if not divergencias.empty:
    print("\nPrimeira divergência encontrada:")
    display(divergencias.iloc[[0]])


,ano,base,n_colunas,colunas_ok,tipos_ok,schema_ok
0,1998,enem,29,True,True,True
1,1999,enem,29,True,True,True
2,2000,enem,29,True,True,True
3,2001,enem,29,True,True,True
4,2002,enem,29,True,True,True
5,2003,enem,29,True,True,True
6,2004,enem,29,True,True,True
7,2005,enem,29,True,True,True
8,2006,enem,29,True,True,True
9,2007,enem,29,True,True,True


Arquivos com esquema correto: 30/30


## 4 — Cardinalidade Silver → Gold

In [4]:

# ============================================================
# 4. CARDINALIDADE SILVER → GOLD
# ============================================================

auditoria_cardinalidade = []

for _, fonte in manifesto_gold.iterrows():

    ano = int(fonte["ano"])
    base = fonte["base"]
    arquivo_silver = str(fonte["arquivo"])
    arquivo_gold = str(fonte["arquivo_gold"])

    n_silver = con.execute(
        """
        SELECT COUNT(*)
        FROM read_parquet(?, hive_partitioning = false)
        """,
        [arquivo_silver]
    ).fetchone()[0]

    n_gold = con.execute(
        """
        SELECT COUNT(*)
        FROM read_parquet(?, hive_partitioning = false)
        """,
        [arquivo_gold]
    ).fetchone()[0]

    auditoria_cardinalidade.append({
        "ano": ano,
        "base": base,
        "n_silver": int(n_silver),
        "n_gold": int(n_gold),
        "diferenca": int(n_gold - n_silver),
        "cardinalidade_ok": n_silver == n_gold
    })

auditoria_cardinalidade = pd.DataFrame(
    auditoria_cardinalidade
)

display(auditoria_cardinalidade)

print("=" * 70)
print("RESULTADO — CARDINALIDADE SILVER → GOLD")
print("=" * 70)

print(
    f"Fontes com cardinalidade preservada: "
    f"{auditoria_cardinalidade['cardinalidade_ok'].sum()}/"
    f"{len(auditoria_cardinalidade)}"
)

print(
    f"Total de linhas Silver: "
    f"{auditoria_cardinalidade['n_silver'].sum():,}"
)

print(
    f"Total de linhas Gold: "
    f"{auditoria_cardinalidade['n_gold'].sum():,}"
)

print(
    f"Diferença total: "
    f"{auditoria_cardinalidade['diferenca'].sum():,}"
)


,ano,base,n_silver,n_gold,diferenca,cardinalidade_ok
0,1998,enem,288,288,0,True
1,1999,enem,6518,6518,0,True
2,2000,enem,6219,6219,0,True
3,2001,enem,43918,43918,0,True
4,2002,enem,36407,36407,0,True
5,2003,enem,44721,44721,0,True
6,2004,enem,36351,36351,0,True
7,2005,enem,72163,72163,0,True
8,2006,enem,104890,104890,0,True
9,2007,enem,101145,101145,0,True


RESULTADO — CARDINALIDADE SILVER → GOLD
Fontes com cardinalidade preservada: 30/30
Total de linhas Silver: 5,841,609
Total de linhas Gold: 5,841,609
Diferença total: 0


## 5 — Referência territorial

In [5]:

# ============================================================
# 5. REFERÊNCIA TERRITORIAL
# ============================================================

def referencia_territorial_esperada(ano):
    if 1998 <= ano <= 2000:
        return "RESIDENCIA"
    elif 2001 <= ano <= 2007:
        return "ESC"
    elif 2008 <= ano <= 2025:
        return "PROVA"
    else:
        raise ValueError(f"Ano fora do intervalo: {ano}")


auditoria_territorial = []

for _, fonte in manifesto_gold.iterrows():
    ano = int(fonte["ano"])
    base = fonte["base"]
    arquivo_gold = str(fonte["arquivo_gold"])

    tipo_esperado = referencia_territorial_esperada(ano)

    resultado = con.execute(
        """
        SELECT
            COUNT(*) AS n_registros,

            COUNT(*) FILTER (
                WHERE TIPO_REFERENCIA_TERRITORIAL = ?
            ) AS n_corretos,

            COUNT(*) FILTER (
                WHERE TIPO_REFERENCIA_TERRITORIAL <> ?
                   OR TIPO_REFERENCIA_TERRITORIAL IS NULL
            ) AS n_incorretos,

            COUNT(DISTINCT TIPO_REFERENCIA_TERRITORIAL)
                AS n_tipos_distintos

        FROM read_parquet(?, hive_partitioning = false)
        """,
        [tipo_esperado, tipo_esperado, arquivo_gold]
    ).fetchone()

    n_registros, n_corretos, n_incorretos, n_tipos = resultado

    auditoria_territorial.append({
        "ano": ano,
        "base": base,
        "referencia_esperada": tipo_esperado,
        "n_registros": int(n_registros),
        "n_corretos": int(n_corretos),
        "n_incorretos": int(n_incorretos),
        "n_tipos_distintos": int(n_tipos),
        "territorio_ok": (
            n_incorretos == 0 and n_tipos == 1
        )
    })

auditoria_territorial = pd.DataFrame(auditoria_territorial)

display(auditoria_territorial)

print("=" * 70)
print("RESULTADO — REFERÊNCIA TERRITORIAL")
print("=" * 70)

print(
    "Fontes com referência territorial correta: "
    f"{auditoria_territorial['territorio_ok'].sum()}/"
    f"{len(auditoria_territorial)}"
)

print(
    "Registros com referência incorreta: "
    f"{auditoria_territorial['n_incorretos'].sum():,}"
)


,ano,base,referencia_esperada,n_registros,n_corretos,n_incorretos,n_tipos_distintos,territorio_ok
0,1998,enem,RESIDENCIA,288,288,0,1,True
1,1999,enem,RESIDENCIA,6518,6518,0,1,True
2,2000,enem,RESIDENCIA,6219,6219,0,1,True
3,2001,enem,ESC,43918,43918,0,1,True
4,2002,enem,ESC,36407,36407,0,1,True
5,2003,enem,ESC,44721,44721,0,1,True
6,2004,enem,ESC,36351,36351,0,1,True
7,2005,enem,ESC,72163,72163,0,1,True
8,2006,enem,ESC,104890,104890,0,1,True
9,2007,enem,ESC,101145,101145,0,1,True


RESULTADO — REFERÊNCIA TERRITORIAL
Fontes com referência territorial correta: 30/30
Registros com referência incorreta: 0


## 6 — Integridade territorial dos registros

In [6]:

# ============================================================
# 6. INTEGRIDADE TERRITORIAL — UF E MUNICÍPIOS
# ============================================================

auditoria_integridade_territorial = []

for _, fonte in manifesto_gold.iterrows():
    ano = int(fonte["ano"])
    base = fonte["base"]
    arquivo_gold = str(fonte["arquivo_gold"])

    resultado = con.execute(
        """
        SELECT
            COUNT(*) AS n_registros,

            COUNT(*) FILTER (
                WHERE NU_ANO = ?
            ) AS n_ano_corretos,

            COUNT(*) FILTER (
                WHERE NU_ANO IS NULL
            ) AS n_ano_nulos,

            COUNT(*) FILTER (
                WHERE SG_UF_TERRITORIO = ?
            ) AS n_sigla_uf_corretos,

            COUNT(*) FILTER (
                WHERE SG_UF_TERRITORIO IS NOT NULL
                  AND SG_UF_TERRITORIO <> ?
            ) AS n_sigla_uf_incorretos,

            COUNT(*) FILTER (
                WHERE SG_UF_TERRITORIO IS NULL
            ) AS n_sigla_uf_nulos,

            COUNT(*) FILTER (
                WHERE CO_UF_TERRITORIO = ?
            ) AS n_codigo_uf_corretos,

            COUNT(*) FILTER (
                WHERE CO_UF_TERRITORIO IS NOT NULL
                  AND CO_UF_TERRITORIO <> ?
            ) AS n_codigo_uf_incorretos,

            COUNT(*) FILTER (
                WHERE CO_UF_TERRITORIO IS NULL
            ) AS n_codigo_uf_nulos,

            COUNT(*) FILTER (
                WHERE CO_MUNICIPIO_TERRITORIO IS NOT NULL
            ) AS n_municipio_informado,

            COUNT(*) FILTER (
                WHERE NO_MUNICIPIO_TERRITORIO IS NOT NULL
            ) AS n_nome_municipio_informado,

            COUNT(DISTINCT CO_UF_TERRITORIO)
                AS n_codigos_uf_distintos,

            COUNT(DISTINCT CO_MUNICIPIO_TERRITORIO)
                AS n_municipios_distintos

        FROM read_parquet(?, hive_partitioning = false)
        """,
        [
            ano,
            UF_PROCESSAMENTO,
            UF_PROCESSAMENTO,
            CODIGO_UF_PROCESSAMENTO,
            CODIGO_UF_PROCESSAMENTO,
            arquivo_gold
        ]
    ).fetchone()

    (
        n_registros,
        n_ano_corretos,
        n_ano_nulos,
        n_sigla_corretos,
        n_sigla_incorretos,
        n_sigla_nulos,
        n_codigo_corretos,
        n_codigo_incorretos,
        n_codigo_nulos,
        n_municipio_informado,
        n_nome_municipio_informado,
        n_codigos_uf_distintos,
        n_municipios_distintos
    ) = resultado

    ano_ok = n_ano_corretos == n_registros

    uf_ok = (
        n_sigla_corretos == n_registros
        and n_codigo_corretos == n_registros
        and n_sigla_incorretos == 0
        and n_sigla_nulos == 0
        and n_codigo_incorretos == 0
        and n_codigo_nulos == 0
    )

    auditoria_integridade_territorial.append({
        "ano": ano,
        "base": base,
        "n_registros": int(n_registros),
        "n_ano_corretos": int(n_ano_corretos),
        "n_ano_nulos": int(n_ano_nulos),
        "n_sigla_uf_corretos": int(n_sigla_corretos),
        "n_sigla_uf_incorretos": int(n_sigla_incorretos),
        "n_sigla_uf_nulos": int(n_sigla_nulos),
        "n_codigo_uf_corretos": int(n_codigo_corretos),
        "n_codigo_uf_incorretos": int(n_codigo_incorretos),
        "n_codigo_uf_nulos": int(n_codigo_nulos),
        "n_municipio_informado": int(n_municipio_informado),
        "n_nome_municipio_informado": int(n_nome_municipio_informado),
        "n_codigos_uf_distintos": int(n_codigos_uf_distintos),
        "n_municipios_distintos": int(n_municipios_distintos),
        "ano_ok": ano_ok,
        "uf_ok": uf_ok
    })

auditoria_integridade_territorial = pd.DataFrame(
    auditoria_integridade_territorial
)

display(auditoria_integridade_territorial)

print("=" * 70)
print("RESULTADO — INTEGRIDADE TERRITORIAL")
print("=" * 70)

print(
    f"Ano consistente: "
    f"{auditoria_integridade_territorial['ano_ok'].sum()}/"
    f"{len(auditoria_integridade_territorial)}"
)

print(
    f"UF {UF_PROCESSAMENTO} consistente: "
    f"{auditoria_integridade_territorial['uf_ok'].sum()}/"
    f"{len(auditoria_integridade_territorial)}"
)

print("\nFontes com problemas territoriais:")
display(
    auditoria_integridade_territorial.loc[
        ~auditoria_integridade_territorial["uf_ok"],
        [
            "ano", "base",
            "n_sigla_uf_incorretos", "n_sigla_uf_nulos",
            "n_codigo_uf_incorretos", "n_codigo_uf_nulos"
        ]
    ]
)


,ano,base,n_registros,n_ano_corretos,n_ano_nulos,n_sigla_uf_corretos,n_sigla_uf_incorretos,n_sigla_uf_nulos,n_codigo_uf_corretos,n_codigo_uf_incorretos,n_codigo_uf_nulos,n_municipio_informado,n_nome_municipio_informado,n_codigos_uf_distintos,n_municipios_distintos,ano_ok,uf_ok
0,1998,enem,288,288,0,288,0,0,288,0,0,288,288,1,5,True,True
1,1999,enem,6518,6518,0,6518,0,0,6518,0,0,6518,6518,1,60,True,True
2,2000,enem,6219,6219,0,6219,0,0,6219,0,0,6219,6219,1,78,True,True
3,2001,enem,43918,43918,0,24263,46,19609,24263,46,19609,24309,24309,13,140,True,False
4,2002,enem,36407,36407,0,32188,68,4151,32188,68,4151,32256,32256,11,155,True,False
5,2003,enem,44721,44721,0,41180,76,3465,41180,76,3465,41256,41256,18,175,True,False
6,2004,enem,36351,36351,0,34668,42,1641,34668,42,1641,34710,34710,12,159,True,False
7,2005,enem,72163,72163,0,49307,50,22806,49307,50,22806,49357,49357,18,174,True,False
8,2006,enem,104890,104890,0,35953,141,68796,35953,141,68796,36094,36094,21,200,True,False
9,2007,enem,101145,101145,0,44608,234,56303,44608,234,56303,44842,44842,16,180,True,False


RESULTADO — INTEGRIDADE TERRITORIAL
Ano consistente: 30/30
UF PA consistente: 23/30

Fontes com problemas territoriais:


,ano,base,n_sigla_uf_incorretos,n_sigla_uf_nulos,n_codigo_uf_incorretos,n_codigo_uf_nulos
3,2001,enem,46,19609,46,19609
4,2002,enem,68,4151,68,4151
5,2003,enem,76,3465,76,3465
6,2004,enem,42,1641,42,1641
7,2005,enem,50,22806,50,22806
8,2006,enem,141,68796,141,68796
9,2007,enem,234,56303,234,56303


### 6A — Diagnosticar as divergências

In [7]:

# ============================================================
# 6A. DIAGNÓSTICO DAS REFERÊNCIAS TERRITORIAIS
# ============================================================

for ano in range(2001, 2008):
    fonte = manifesto_gold.loc[
        (manifesto_gold["ano"] == ano)
        & (manifesto_gold["base"] == "enem")
    ].iloc[0]

    arquivo_silver = str(fonte["arquivo"])
    arquivo_gold = str(fonte["arquivo_gold"])

    print("\n" + "=" * 70)
    print(f"ANO {ano}")
    print("=" * 70)

    # Colunas territoriais disponíveis na Silver
    schema_silver = con.execute(
        """
        DESCRIBE
        SELECT *
        FROM read_parquet(?, hive_partitioning = false)
        """,
        [arquivo_silver]
    ).df()

    colunas_silver = schema_silver["column_name"].tolist()

    colunas_relevantes = [
        coluna for coluna in colunas_silver
        if any(
            termo in coluna.upper()
            for termo in (
                "UF", "MUNICIPIO", "TERRITORIO",
                "RESIDENCIA", "ESC", "PROVA"
            )
        )
    ]

    print("Colunas territoriais na Silver:")
    print(colunas_relevantes)

    # Situação dos campos territoriais na Gold
    diagnostico = con.execute(
        """
        SELECT
            COUNT(*) AS total,

            COUNT(SG_UF_TERRITORIO) AS sigla_preenchida,
            COUNT(CO_UF_TERRITORIO) AS codigo_uf_preenchido,
            COUNT(CO_MUNICIPIO_TERRITORIO)
                AS codigo_municipio_preenchido,
            COUNT(NO_MUNICIPIO_TERRITORIO)
                AS nome_municipio_preenchido,

            COUNT(DISTINCT SG_UF_TERRITORIO)
                AS siglas_distintas,
            COUNT(DISTINCT CO_UF_TERRITORIO)
                AS codigos_uf_distintos

        FROM read_parquet(?, hive_partitioning = false)
        """,
        [arquivo_gold]
    ).df()

    display(diagnostico)

    # Distribuição dos valores de UF existentes
    distribuicao = con.execute(
        """
        SELECT
            SG_UF_TERRITORIO,
            CO_UF_TERRITORIO,
            COUNT(*) AS registros
        FROM read_parquet(?, hive_partitioning = false)
        GROUP BY 1, 2
        ORDER BY registros DESC
        LIMIT 15
        """,
        [arquivo_gold]
    ).df()

    print("Distribuição dos campos de UF na Gold:")
    display(distribuicao)



ANO 2001
Colunas territoriais na Silver:
['CO_MUNICIPIO_RESIDENCIA', 'NO_MUNICIPIO_RESIDENCIA', 'CO_UF_RESIDENCIA', 'SG_UF_RESIDENCIA', 'CO_MUNICIPIO_ESC', 'NO_MUNICIPIO_ESC', 'CO_UF_ESC', 'SG_UF_ESC', 'TP_DEPENDENCIA_ADM_ESC', 'TP_LOCALIZACAO_ESC', 'TP_SIT_FUNC_ESC', 'CO_PROVA']


,total,sigla_preenchida,codigo_uf_preenchido,codigo_municipio_preenchido,nome_municipio_preenchido,siglas_distintas,codigos_uf_distintos
0,43918,24309,24309,24309,24309,13,13


Distribuição dos campos de UF na Gold:


,SG_UF_TERRITORIO,CO_UF_TERRITORIO,registros
0,PA,15,24263
1,None,None,19609
2,TO,17,14
3,MA,21,12
4,AP,16,3
5,AM,13,3
6,CE,23,3
7,MG,31,2
8,SP,35,2
9,PR,41,2



ANO 2002
Colunas territoriais na Silver:
['CO_MUNICIPIO_RESIDENCIA', 'NO_MUNICIPIO_RESIDENCIA', 'CO_UF_RESIDENCIA', 'SG_UF_RESIDENCIA', 'CO_MUNICIPIO_ESC', 'NO_MUNICIPIO_ESC', 'CO_UF_ESC', 'SG_UF_ESC', 'TP_DEPENDENCIA_ADM_ESC', 'TP_LOCALIZACAO_ESC', 'TP_SIT_FUNC_ESC', 'CO_PROVA']


,total,sigla_preenchida,codigo_uf_preenchido,codigo_municipio_preenchido,nome_municipio_preenchido,siglas_distintas,codigos_uf_distintos
0,36407,32256,32256,32256,32256,11,11


... (18 outputs left out)


ANO 2007
Colunas territoriais na Silver:
['CO_MUNICIPIO_RESIDENCIA', 'NO_MUNICIPIO_RESIDENCIA', 'CO_UF_RESIDENCIA', 'SG_UF_RESIDENCIA', 'CO_MUNICIPIO_ESC', 'NO_MUNICIPIO_ESC', 'CO_UF_ESC', 'SG_UF_ESC', 'TP_DEPENDENCIA_ADM_ESC', 'TP_LOCALIZACAO_ESC', 'TP_SIT_FUNC_ESC', 'CO_PROVA']


,total,sigla_preenchida,codigo_uf_preenchido,codigo_municipio_preenchido,nome_municipio_preenchido,siglas_distintas,codigos_uf_distintos
0,101145,44842,44842,44842,44842,16,16


Distribuição dos campos de UF na Gold:


,SG_UF_TERRITORIO,CO_UF_TERRITORIO,registros
0,None,None,56303
1,PA,15,44608
2,MA,21,196
3,TO,17,9
4,SP,35,5
5,AM,13,5
6,GO,52,4
7,MG,31,3
8,AP,16,2
9,CE,23,2


### 6B — Comparar os campos de origem com a Gold

In [8]:

# ============================================================
# 6B. COMPARAÇÃO TERRITORIAL SILVER → GOLD (2001–2007)
# ============================================================

for ano in range(2001, 2008):

    fonte = manifesto_gold.loc[
        (manifesto_gold["ano"] == ano)
        & (manifesto_gold["base"] == "enem")
    ].iloc[0]

    arquivo_silver = str(fonte["arquivo"])
    arquivo_gold = str(fonte["arquivo_gold"])

    # Campos de escola na Silver
    silver = con.execute(
        """
        SELECT
            COUNT(*) AS total,
            COUNT(SG_UF_ESC) AS sigla_escola,
            COUNT(CO_UF_ESC) AS codigo_uf_escola,
            COUNT(CO_MUNICIPIO_ESC) AS codigo_municipio_escola,
            COUNT(NO_MUNICIPIO_ESC) AS nome_municipio_escola
        FROM read_parquet(?, hive_partitioning = false)
        """,
        [arquivo_silver]
    ).df()

    # Campos canônicos na Gold
    gold = con.execute(
        """
        SELECT
            COUNT(*) AS total,
            COUNT(SG_UF_TERRITORIO) AS sigla_territorio,
            COUNT(CO_UF_TERRITORIO) AS codigo_uf_territorio,
            COUNT(CO_MUNICIPIO_TERRITORIO) AS codigo_municipio_territorio,
            COUNT(NO_MUNICIPIO_TERRITORIO) AS nome_municipio_territorio
        FROM read_parquet(?, hive_partitioning = false)
        """,
        [arquivo_gold]
    ).df()

    print(f"\nANO {ano}")
    print("Silver — referência escolar:")
    display(silver)

    print("Gold — referência territorial:")
    display(gold)



ANO 2001
Silver — referência escolar:


,total,sigla_escola,codigo_uf_escola,codigo_municipio_escola,nome_municipio_escola
0,43918,24309,24309,24309,24309


Gold — referência territorial:


,total,sigla_territorio,codigo_uf_territorio,codigo_municipio_territorio,nome_municipio_territorio
0,43918,24309,24309,24309,24309



ANO 2002
Silver — referência escolar:


,total,sigla_escola,codigo_uf_escola,codigo_municipio_escola,nome_municipio_escola
0,36407,32256,32256,32256,32256


... (18 outputs left out)


ANO 2007
Silver — referência escolar:


,total,sigla_escola,codigo_uf_escola,codigo_municipio_escola,nome_municipio_escola
0,101145,44842,44842,44842,44842


Gold — referência territorial:


,total,sigla_territorio,codigo_uf_territorio,codigo_municipio_territorio,nome_municipio_territorio
0,101145,44842,44842,44842,44842


### 6C — Diagnóstico da referência territorial

In [9]:
for ano in range(2001, 2008):
    fonte = manifesto_gold.loc[
        (manifesto_gold["ano"] == ano)
        & (manifesto_gold["base"] == "enem")
    ].iloc[0]

    arquivo_silver = str(fonte["arquivo"])

    diagnostico = con.execute("""
        SELECT
            SG_UF_ESC,
            CO_UF_ESC,
            COUNT(*) AS quantidade,
            COUNT(CO_MUNICIPIO_ESC) AS com_codigo_municipio,
            COUNT(NO_MUNICIPIO_ESC) AS com_nome_municipio
        FROM read_parquet(?, hive_partitioning = false)
        GROUP BY SG_UF_ESC, CO_UF_ESC
        ORDER BY quantidade DESC
    """, [arquivo_silver]).df()

    print(f"\nANO {ano}")
    display(diagnostico)


ANO 2001


,SG_UF_ESC,CO_UF_ESC,quantidade,com_codigo_municipio,com_nome_municipio
0,PA,15,24263,24263,24263
1,None,<NA>,19609,0,0
2,TO,17,14,14,14
3,MA,21,12,12,12
4,AM,13,3,3,3
5,CE,23,3,3,3
6,AP,16,3,3,3
7,SP,35,2,2,2
8,MG,31,2,2,2
9,RO,11,2,2,2



ANO 2002


,SG_UF_ESC,CO_UF_ESC,quantidade,com_codigo_municipio,com_nome_municipio
0,PA,15,32188,32188,32188
1,None,<NA>,4151,0,0
2,MA,21,19,19,19
3,AM,13,15,15,15
4,TO,17,14,14,14
5,SP,35,7,7,7
6,PI,22,5,5,5
7,PR,41,3,3,3
8,CE,23,2,2,2
9,ES,32,1,1,1



ANO 2003


,SG_UF_ESC,CO_UF_ESC,quantidade,com_codigo_municipio,com_nome_municipio
0,PA,15,41180,41180,41180
1,None,<NA>,3465,0,0
2,TO,17,14,14,14
3,MA,21,14,14,14
4,AM,13,13,13,13
5,SE,28,9,9,9
6,SP,35,7,7,7
7,RJ,33,6,6,6
8,SC,42,2,2,2
9,RO,11,2,2,2


... (4 outputs left out)


ANO 2006


,SG_UF_ESC,CO_UF_ESC,quantidade,com_codigo_municipio,com_nome_municipio
0,None,<NA>,68796,0,0
1,PA,15,35953,35953,35953
2,SP,35,49,49,49
3,AM,13,16,16,16
4,TO,17,11,11,11
5,MT,51,9,9,9
6,RJ,33,8,8,8
7,AP,16,7,7,7
8,GO,52,6,6,6
9,MA,21,6,6,6



ANO 2007


,SG_UF_ESC,CO_UF_ESC,quantidade,com_codigo_municipio,com_nome_municipio
0,None,<NA>,56303,0,0
1,PA,15,44608,44608,44608
2,MA,21,196,196,196
3,TO,17,9,9,9
4,AM,13,5,5,5
5,SP,35,5,5,5
6,GO,52,4,4,4
7,MG,31,3,3,3
8,CE,23,2,2,2
9,AP,16,2,2,2


### 6D

In [10]:
for ano in range(2001, 2008):
    fonte = manifesto_gold.loc[
        (manifesto_gold["ano"] == ano)
        & (manifesto_gold["base"] == "enem")
    ].iloc[0]

    arquivo_silver = str(fonte["arquivo"])

    diagnostico = con.execute("""
        SELECT
            SG_UF_RESIDENCIA,
            CO_UF_RESIDENCIA,
            SG_UF_ESC,
            CO_UF_ESC,
            COUNT(*) AS quantidade
        FROM read_parquet(?, hive_partitioning = false)
        GROUP BY
            SG_UF_RESIDENCIA,
            CO_UF_RESIDENCIA,
            SG_UF_ESC,
            CO_UF_ESC
        ORDER BY quantidade DESC
        LIMIT 15
    """, [arquivo_silver]).df()

    print(f"ANO {ano}")
    display(diagnostico)

ANO 2001


,SG_UF_RESIDENCIA,CO_UF_RESIDENCIA,SG_UF_ESC,CO_UF_ESC,quantidade
0,PA,15,PA,15,24263
1,PA,15,None,<NA>,19609
2,PA,15,TO,17,14
3,PA,15,MA,21,12
4,PA,15,AP,16,3
5,PA,15,AM,13,3
6,PA,15,CE,23,3
7,PA,15,SP,35,2
8,PA,15,PR,41,2
9,PA,15,RO,11,2


ANO 2002


,SG_UF_RESIDENCIA,CO_UF_RESIDENCIA,SG_UF_ESC,CO_UF_ESC,quantidade
0,PA,15,PA,15,32188
1,PA,15,None,<NA>,4151
2,PA,15,MA,21,19
3,PA,15,AM,13,15
4,PA,15,TO,17,14
5,PA,15,SP,35,7
6,PA,15,PI,22,5
7,PA,15,PR,41,3
8,PA,15,CE,23,2
9,PA,15,AP,16,1


ANO 2003


,SG_UF_RESIDENCIA,CO_UF_RESIDENCIA,SG_UF_ESC,CO_UF_ESC,quantidade
0,PA,15,PA,15,41180
1,PA,15,None,<NA>,3465
2,PA,15,TO,17,14
3,PA,15,MA,21,14
4,PA,15,AM,13,13
5,PA,15,SE,28,9
6,PA,15,SP,35,7
7,PA,15,RJ,33,6
8,PA,15,RO,11,2
9,PA,15,SC,42,2


... (4 outputs left out)

ANO 2006


,SG_UF_RESIDENCIA,CO_UF_RESIDENCIA,SG_UF_ESC,CO_UF_ESC,quantidade
0,PA,15,None,<NA>,68796
1,PA,15,PA,15,35953
2,PA,15,SP,35,49
3,PA,15,AM,13,16
4,PA,15,TO,17,11
5,PA,15,MT,51,9
6,PA,15,RJ,33,8
7,PA,15,AP,16,7
8,PA,15,MA,21,6
9,PA,15,PR,41,6


ANO 2007


,SG_UF_RESIDENCIA,CO_UF_RESIDENCIA,SG_UF_ESC,CO_UF_ESC,quantidade
0,PA,15,None,<NA>,53705
1,PA,15,PA,15,43036
2,PA,<NA>,None,<NA>,2457
3,PA,<NA>,PA,15,1539
4,PA,21,MA,21,190
5,PA,21,None,<NA>,57
6,PA,35,None,<NA>,8
7,PA,21,PA,15,8
8,PA,22,None,<NA>,8
9,PA,13,None,<NA>,8


### 6E — Diagnóstico da residência em 2007

In [11]:
fonte = manifesto_gold.loc[
    (manifesto_gold["ano"] == 2007)
    & (manifesto_gold["base"] == "enem")
].iloc[0]

arquivo_silver = str(fonte["arquivo"])

diagnostico_residencia = con.execute("""
    SELECT
        SG_UF_RESIDENCIA,
        CO_UF_RESIDENCIA,
        COUNT(*) AS quantidade
    FROM read_parquet(?, hive_partitioning = false)
    GROUP BY
        SG_UF_RESIDENCIA,
        CO_UF_RESIDENCIA
    ORDER BY quantidade DESC
""", [arquivo_silver]).df()

display(diagnostico_residencia)

,SG_UF_RESIDENCIA,CO_UF_RESIDENCIA,quantidade
0,PA,15,96768
1,PA,<NA>,4001
2,PA,21,255
3,PA,13,13
4,PA,52,12
5,PA,35,11
6,PA,31,10
7,PA,25,10
8,PA,16,9
9,PA,22,9


## 7 — Compatibilidade municipal de 2024–2025

In [12]:
for ano in [2024, 2025]:
    fontes = manifesto_gold.loc[
        (manifesto_gold["ano"] == ano)
        & (manifesto_gold["base"].isin(["participantes", "resultados"]))
    ]

    if fontes.empty:
        print(f"{ano}: fontes não encontradas no manifesto.")
        continue

    print(f"\nANO {ano}")

    for _, fonte in fontes.iterrows():
        arquivo = str(fonte["arquivo"])

        resumo = con.execute("""
            SELECT
                COUNT(*) AS total_registros,
                COUNT(DISTINCT CO_MUNICIPIO_PROVA) AS municipios,
                COUNT(*) FILTER (
                    WHERE CO_MUNICIPIO_PROVA IS NULL
                ) AS municipio_nulo
            FROM read_parquet(?, hive_partitioning = false)
        """, [arquivo]).df()

        print(f"Base: {fonte['base']}")
        display(resumo)


ANO 2024
Base: participantes


,total_registros,municipios,municipio_nulo
0,248061,81,0


Base: resultados


,total_registros,municipios,municipio_nulo
0,248061,81,0



ANO 2025
Base: participantes


,total_registros,municipios,municipio_nulo
0,289328,89,0


Base: resultados


,total_registros,municipios,municipio_nulo
0,289328,89,0


In [13]:
for ano in [2024, 2025]:
    fontes = manifesto_gold.loc[
        (manifesto_gold["ano"] == ano)
        & (manifesto_gold["base"].isin(["participantes", "resultados"]))
    ]

    consultas = []

    for _, fonte in fontes.iterrows():
        base = fonte["base"]
        arquivo = str(fonte["arquivo"])

        df = con.execute("""
            SELECT
                CO_MUNICIPIO_PROVA,
                NO_MUNICIPIO_PROVA,
                COUNT(*) AS quantidade
            FROM read_parquet(?, hive_partitioning = false)
            GROUP BY CO_MUNICIPIO_PROVA, NO_MUNICIPIO_PROVA
        """, [arquivo]).df()

        df["base"] = base
        consultas.append(df)

    comparacao = pd.concat(consultas, ignore_index=True)

    tabela = comparacao.pivot_table(
        index=["CO_MUNICIPIO_PROVA", "NO_MUNICIPIO_PROVA"],
        columns="base",
        values="quantidade",
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    tabela["diferenca"] = (
        tabela.get("participantes", 0) - tabela.get("resultados", 0)
    )

    print(f"\nANO {ano}")
    display(tabela)
    print("Municípios com divergência:", (tabela["diferenca"] != 0).sum())


ANO 2024


base,CO_MUNICIPIO_PROVA,NO_MUNICIPIO_PROVA,participantes,resultados,diferenca
0,1500107,Abaetetuba,6601,6601,0
1,1500206,Acará,1031,1031,0
2,1500404,Alenquer,1595,1595,0
3,1500503,Almeirim,506,506,0
4,1500602,Altamira,5888,5888,0
...,...,...,...,...,...
76,1508100,Tucuruí,3141,3141,0
77,1508159,Uruará,888,888,0
78,1508209,Vigia,2238,2238,0
79,1508308,Viseu,1237,1237,0


Municípios com divergência: 0

ANO 2025


base,CO_MUNICIPIO_PROVA,NO_MUNICIPIO_PROVA,participantes,resultados,diferenca
0,1500107,Abaetetuba,7485,7485,0
1,1500206,Acará,1369,1369,0
2,1500404,Alenquer,1856,1856,0
3,1500503,Almeirim,582,582,0
4,1500602,Altamira,5351,5351,0
...,...,...,...,...,...
84,1508159,Uruará,889,889,0
85,1508209,Vigia,2751,2751,0
86,1508308,Viseu,1339,1339,0
87,1508357,Vitória do Xingu,466,466,0


Municípios com divergência: 0


## 8 — Perfil de preenchimento

In [14]:
for _, fonte in manifesto_gold.iterrows():
    ano = int(fonte["ano"])
    arquivo_gold = str(fonte["arquivo_gold"])

    perfil = con.execute("""
        SELECT *
        FROM read_parquet(?, hive_partitioning = false)
    """, [arquivo_gold]).df()

    colunas = [
        "NU_NOTA_REDACAO",
        "NU_NOTA_COMP1",
        "NU_NOTA_COMP2",
        "NU_NOTA_COMP3",
        "NU_NOTA_COMP4",
        "NU_NOTA_COMP5",
        "TP_SEXO",
        "TP_FAIXA_ETARIA",
        "TP_STATUS_REDACAO",
        "SG_UF_TERRITORIO",
        "CO_MUNICIPIO_TERRITORIO",
    ]

    resultado = {
        "ano": ano,
        "base": fonte["base"],
        "registros": len(perfil),
    }

    for coluna in colunas:
        if coluna in perfil.columns:
            resultado[f"{coluna}_pct"] = round(
                perfil[coluna].notna().mean() * 100, 2
            )

    print(resultado)

{'ano': 1998, 'base': 'enem', 'registros': 288, 'NU_NOTA_REDACAO_pct': 100.0, 'TP_SEXO_pct': 100.0, 'TP_FAIXA_ETARIA_pct': 100.0, 'TP_STATUS_REDACAO_pct': 100.0, 'SG_UF_TERRITORIO_pct': 100.0, 'CO_MUNICIPIO_TERRITORIO_pct': 100.0}
{'ano': 1999, 'base': 'enem', 'registros': 6518, 'NU_NOTA_REDACAO_pct': 100.0, 'TP_SEXO_pct': 100.0, 'TP_FAIXA_ETARIA_pct': 100.0, 'TP_STATUS_REDACAO_pct': 100.0, 'SG_UF_TERRITORIO_pct': 100.0, 'CO_MUNICIPIO_TERRITORIO_pct': 100.0}
{'ano': 2000, 'base': 'enem', 'registros': 6219, 'NU_NOTA_REDACAO_pct': 100.0, 'TP_SEXO_pct': 100.0, 'TP_FAIXA_ETARIA_pct': 100.0, 'TP_STATUS_REDACAO_pct': 100.0, 'SG_UF_TERRITORIO_pct': 100.0, 'CO_MUNICIPIO_TERRITORIO_pct': 100.0}
{'ano': 2001, 'base': 'enem', 'registros': 43918, 'NU_NOTA_REDACAO_pct': 100.0, 'TP_SEXO_pct': 100.0, 'TP_FAIXA_ETARIA_pct': 100.0, 'TP_STATUS_REDACAO_pct': 100.0, 'SG_UF_TERRITORIO_pct': 55.35, 'CO_MUNICIPIO_TERRITORIO_pct': 55.35}
{'ano': 2002, 'base': 'enem', 'registros': 36407, 'NU_NOTA_REDACAO_pct':

## 9 — Relatório final

In [15]:
print("=" * 60)
print(f"AUDITORIA GOLD — UF {UF_PROCESSAMENTO}")
print("=" * 60)

print(f"Fontes auditadas: {len(manifesto_gold)}")
print(f"Período: {ANO_INICIAL}–{ANO_FINAL}")
print(f"Anos cobertos: {manifesto_gold['ano'].nunique()}")

print("\nRESULTADOS")
print("- Integridade do manifesto: aprovada")
print("- Esquema canônico: aprovado")
print("- Cardinalidade Silver → Gold: aprovada")
print("- Referência territorial por período: aprovada")
print("- Compatibilidade PARTICIPANTES × RESULTADOS (2024–2025): aprovada")

print("\nPONTOS DE ATENÇÃO")
print("- Completude e consistência territorial em 2001–2007.")
print("- Preenchimento variável das notas e do status da redação.")
print("- Diferenças de colunas entre PARTICIPANTES e RESULTADOS em 2024–2025.")

print("\nConclusão:")
print("A estrutura e a cardinalidade da Gold foram validadas.")
print("As ressalvas de qualidade dos dados devem ser documentadas.")

AUDITORIA GOLD — UF PA
Fontes auditadas: 30
Período: 1998–2025
Anos cobertos: 28

RESULTADOS
- Integridade do manifesto: aprovada
- Esquema canônico: aprovado
- Cardinalidade Silver → Gold: aprovada
- Referência territorial por período: aprovada
- Compatibilidade PARTICIPANTES × RESULTADOS (2024–2025): aprovada

PONTOS DE ATENÇÃO
- Completude e consistência territorial em 2001–2007.
- Preenchimento variável das notas e do status da redação.
- Diferenças de colunas entre PARTICIPANTES e RESULTADOS em 2024–2025.

Conclusão:
A estrutura e a cardinalidade da Gold foram validadas.
As ressalvas de qualidade dos dados devem ser documentadas.
